# Python Package Management and Publishing with `pip`

This notebook is a **from basics to high intermediate** guide to packaging Python projects so other people can install them with **`pip`**.

We will focus on the modern workflow around **`pyproject.toml`**, wheels, source distributions, virtual environments, editable installs, TestPyPI / PyPI publication, authentication, versioning, private indexes, and release management.

Most examples build a tiny package inside a **temporary directory**, so you can inspect a realistic project without cluttering your repo.

### Contents

- [1. Packaging terms: package, distribution, `pip`, PyPI](#1-packaging-terms-package-distribution-pip-pypi)
- [2. Minimal project layout with `pyproject.toml`](#2-minimal-project-layout-with-pyprojecttoml)
- [3. Metadata, dependencies, and entry points](#3-metadata-dependencies-and-entry-points)
- [4. Building wheels and source distributions](#4-building-wheels-and-source-distributions)
- [5. Local installs, editable mode, and virtual environments](#5-local-installs-editable-mode-and-virtual-environments)
- [6. Publishing to TestPyPI and PyPI](#6-publishing-to-testpypi-and-pypi)
- [7. Authentication, authorization, and `.pypirc`](#7-authentication-authorization-and-pypirc)
- [8. Updating releases and versioning](#8-updating-releases-and-versioning)
- [9. Private indexes and dependency sources](#9-private-indexes-and-dependency-sources)
- [10. Managing releases: yank, replace, transfer, remove](#10-managing-releases-yank-replace-transfer-remove)

In [ ]:
from __future__ import annotations

import configparser
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import textwrap
import venv

try:
    import tomllib
except ModuleNotFoundError:
    import tomli as tomllib

WORKDIR = Path(tempfile.mkdtemp(prefix="g_pip_"))
print("Notebook workspace:", WORKDIR)


def run(cmd, cwd=None, env=None):
    print("$", " ".join(map(str, cmd)))
    completed = subprocess.run(cmd, cwd=cwd, env=env, text=True, capture_output=True)
    if completed.stdout.strip():
        print(completed.stdout.strip())
    if completed.stderr.strip():
        print(completed.stderr.strip())
    print("return code:", completed.returncode)
    return completed


def show_tree(root: Path, max_depth: int = 3) -> None:
    root = Path(root)
    for path in sorted(root.rglob("*")):
        depth = len(path.relative_to(root).parts)
        if depth > max_depth:
            continue
        indent = "  " * (depth - 1)
        suffix = "/" if path.is_dir() else ""
        print(f"{indent}{path.name}{suffix}")


def write_demo_project(root: Path, version: str = "0.1.0") -> Path:
    root = Path(root)
    src = root / "src" / "demo_pkg"
    src.mkdir(parents=True, exist_ok=True)

    (root / "README.md").write_text(
        "# demo-pkg\n\nA tiny package created inside g_pip.ipynb.\n",
        encoding="utf-8",
    )
    (src / "__init__.py").write_text(
        textwrap.dedent(
            f'''
            __all__ = ["__version__", "hello", "meaning_of_packaging"]
            __version__ = "{version}"

            def hello(name: str = "world") -> str:
                return f"hello, {{name}}"

            def meaning_of_packaging() -> str:
                return "Package once, install reproducibly."
            '''
        ).strip()
        + "\n",
        encoding="utf-8",
    )
    (src / "cli.py").write_text(
        textwrap.dedent(
            """
            from . import hello

            def main() -> None:
                print(hello("from console script"))
            """
        ).strip()
        + "\n",
        encoding="utf-8",
    )
    (root / "pyproject.toml").write_text(
        textwrap.dedent(
            f"""
            [build-system]
            requires = ["setuptools>=68", "wheel"]
            build-backend = "setuptools.build_meta"

            [project]
            name = "demo-pkg-cursor"
            version = "{version}"
            description = "Temporary demo package for packaging examples"
            readme = "README.md"
            requires-python = ">=3.10"
            authors = [{{name = "Notebook Demo"}}]
            dependencies = []

            [project.optional-dependencies]
            dev = ["pytest>=8", "ruff>=0.6"]
            docs = ["mkdocs>=1.6"]

            [project.scripts]
            demo-hello = "demo_pkg.cli:main"

            [tool.setuptools]
            package-dir = {{"" = "src"}}

            [tool.setuptools.packages.find]
            where = ["src"]
            """
        ).strip()
        + "\n",
        encoding="utf-8",
    )
    return root


DEMO = write_demo_project(WORKDIR / "demo_pkg_project")
print("Created demo project:")
show_tree(DEMO)

## 1. Packaging terms: package, distribution, `pip`, PyPI

These words are often mixed together, but they are not identical:

- An **import package** is what Python imports, such as `demo_pkg`.
- A **distribution** is the installable project published to an index, such as `demo-pkg-cursor`.
- **`pip`** is the installer / dependency resolver.
- **PyPI** is the public package index that `pip` talks to by default.

In casual speech people say they "upload to pip", but technically you upload a **distribution to PyPI** and then `pip install ...` downloads it from an index.

It also helps to distinguish:

- **Wheel**: built artifact, usually fastest to install.
- **sdist**: source distribution, built on the target machine.
- **Version**: immutable release identifier on PyPI once uploaded.

In [ ]:
print("Python executable:", sys.executable)
run([sys.executable, "-m", "pip", "--version"])

terms = {
    "import package": "demo_pkg",
    "distribution": "demo-pkg-cursor",
    "installer": "pip",
    "index": "PyPI (or another simple index)",
    "built artifact": "wheel (.whl)",
    "source artifact": "sdist (.tar.gz)",
}

for key, value in terms.items():
    print(f"{key:15} -> {value}")

## 2. Minimal project layout with `pyproject.toml`

Modern Python packaging centers on **`pyproject.toml`**. It declares:

- the **build backend** (`setuptools`, `hatchling`, `poetry-core`, etc.),
- project **metadata** (name, version, Python requirement, authors),
- runtime **dependencies**,
- optional dependency groups,
- console scripts / entry points.

A common layout is the **`src/` layout**:

- `pyproject.toml`
- `README.md`
- `src/demo_pkg/__init__.py`
- `src/demo_pkg/...`

The `src/` layout reduces accidental imports from the project root and better matches how installed packages are discovered.

In [ ]:
print("Project tree:")
show_tree(DEMO)

print("\npyproject.toml:\n")
print((DEMO / "pyproject.toml").read_text(encoding="utf-8"))

print("\nsrc/demo_pkg/__init__.py:\n")
print((DEMO / "src" / "demo_pkg" / "__init__.py").read_text(encoding="utf-8"))

## 3. Metadata, dependencies, and entry points

The `[project]` table is the core metadata that packaging tools and package indexes read.

Important fields include:

- `name`: the **distribution** name users install with `pip`.
- `version`: the release number.
- `requires-python`: supported interpreter range.
- `dependencies`: runtime requirements.
- `optional-dependencies`: extras such as `dev`, `docs`, or `postgres`.
- `scripts`: console commands created during installation.

A useful subtlety: the **distribution name** and the **import package name** do not need to be identical, although keeping them close reduces confusion.

In [ ]:
data = tomllib.loads((DEMO / "pyproject.toml").read_text(encoding="utf-8"))
project = data["project"]

print("Distribution name:", project["name"])
print("Version:", project["version"])
print("Requires Python:", project["requires-python"])
print("Dependencies:", project["dependencies"])
print("Optional dependency groups:", list(project["optional-dependencies"].keys()))
print("Console scripts:", project["scripts"])
print("Import package:", "demo_pkg")

## 4. Building wheels and source distributions

Before you publish a package, you normally build artifacts locally and inspect them.

Two common formats are:

- **Wheel (`.whl`)**: pre-built installable artifact.
- **sdist (`.tar.gz`)**: source archive used to build on the target machine.

Typical modern commands are:

- `python -m build --sdist --wheel`
- `python -m pip wheel --no-deps . -w dist`

`build` is the most explicit packaging tool for creating both artifact types. `pip wheel` is also useful, especially in CI or when you want a quick local wheel build.

In [ ]:
dist_dir = DEMO / "dist"
shutil.rmtree(dist_dir, ignore_errors=True)

print("Build a wheel with pip:")
run(
    [
        sys.executable,
        "-m",
        "pip",
        "wheel",
        "--no-deps",
        "--no-build-isolation",
        ".",
        "-w",
        "dist",
    ],
    cwd=DEMO,
)

print("\nTry building both wheel and sdist with python -m build:")
build_result = run(
    [sys.executable, "-m", "build", "--sdist", "--wheel", "--outdir", "dist"],
    cwd=DEMO,
)
if build_result.returncode != 0:
    print("`python -m build` is optional. Install it with `python -m pip install build` if you want both sdist and wheel locally.")

print("\nArtifacts in dist/:")
if dist_dir.exists():
    for path in sorted(dist_dir.iterdir()):
        print("-", path.name)
else:
    print("dist/ was not created")

## 5. Local installs, editable mode, and virtual environments

A good packaging workflow tests installation in a **clean virtual environment**.

Useful patterns:

- `python -m venv .venv` to isolate dependencies.
- `python -m pip install .` for a normal local install.
- `python -m pip install -e .` for an **editable** install while developing.

Editable installs are ideal when the code changes often: the environment points back to your source tree, so you do not rebuild and reinstall on every small edit.

For release testing, however, also test the **built wheel** in a clean environment, because that is closer to what end users receive.

In [ ]:
venv_dir = WORKDIR / "venv_demo"
if not venv_dir.exists():
    venv.EnvBuilder(with_pip=True).create(venv_dir)

venv_python = venv_dir / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
print("Virtual environment Python:", venv_python)

run([str(venv_python), "-m", "pip", "--version"])
install_result = run(
    [str(venv_python), "-m", "pip", "install", "--no-build-isolation", "-e", str(DEMO)]
)
if install_result.returncode == 0:
    run(
        [
            str(venv_python),
            "-c",
            "import demo_pkg; print(demo_pkg.__version__); print(demo_pkg.hello('editable install'))",
        ]
    )
else:
    print("Editable install failed in this environment; the command above is still the right workflow to test in a clean venv.")

## 6. Publishing to TestPyPI and PyPI

To make your package installable with `pip`, the usual publication flow is:

1. build artifacts locally,
2. upload them to **TestPyPI** first,
3. install from TestPyPI in a fresh environment,
4. upload the exact same version to **PyPI**.

Common tools and ideas:

- **`twine upload dist/*`** to upload built files.
- **TestPyPI** for dress rehearsals.
- **Trusted publishing** in CI for tokenless uploads from a verified workflow.
- `pip install --index-url ... --extra-index-url ...` to test from a non-default index.

The key mental model is: **you upload distributions to an index; users later install them with `pip`.**

In [ ]:
run([sys.executable, "-m", "twine", "--version"])

publish_commands = [
    "python -m build --sdist --wheel",
    "python -m twine upload --repository testpypi dist/*",
    "python -m pip install --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple demo-pkg-cursor",
    "python -m twine upload dist/*",
]

print("Recommended publish flow:")
for command in publish_commands:
    print("-", command)

print("\nIf you use GitHub Actions trusted publishing, PyPI can trust the workflow identity instead of a long-lived API token.")

## 7. Authentication, authorization, and `.pypirc`

Publishing needs **authentication** and often some form of **authorization**:

- **Authentication** proves who you are (API token, trusted publishing identity).
- **Authorization** determines what you are allowed to do (owner, maintainer, organization permissions).

Three common approaches are:

- **PyPI API tokens** stored in environment variables or a secret manager.
- **Trusted publishing** from CI using OpenID Connect.
- A local **`.pypirc`** file for tool configuration.

For teams, keep in mind that package maintenance is not just technical. You may need to add maintainers, transfer project ownership, rotate tokens, or revoke compromised credentials.

In [ ]:
pypirc_path = WORKDIR / ".pypirc.example"
pypirc_path.write_text(
    textwrap.dedent(
        """
        [distutils]
        index-servers =
            pypi
            testpypi

        [pypi]
        repository = https://upload.pypi.org/legacy/
        username = __token__
        password = pypi-AgEIcHlwaS5vcmcCJDEyMzQ1Njc4OTBhYmNkZWY=

        [testpypi]
        repository = https://test.pypi.org/legacy/
        username = __token__
        password = pypi-AgEIcHlwaS5vcmcCJGFiY2RlZjAxMjM0NTY3ODk=
        """
    ).strip()
    + "\n",
    encoding="utf-8",
)

parser = configparser.ConfigParser()
parser.read(pypirc_path, encoding="utf-8")

print("Wrote example config:", pypirc_path)
for repo_name in ("pypi", "testpypi"):
    section = parser[repo_name]
    masked = section["password"][:12] + "..."
    print(repo_name, "->", section["repository"], "as", section["username"], "token", masked)

print("\nReal projects should prefer environment variables or trusted publishing over committing secrets.")

## 8. Updating releases and versioning

Once a version is published to PyPI, treat it as **immutable**. The normal fix for a bad release is **publish a newer version**, not overwrite the old files.

That makes versioning discipline important:

- bump versions before each release,
- tag releases in Git,
- document breaking changes,
- keep changelogs or release notes.

Many teams use a **SemVer-like** scheme:

- `MAJOR` for breaking changes,
- `MINOR` for new backward-compatible features,
- `PATCH` for fixes.

Even if your versioning style is simpler, avoid reusing version numbers for different artifacts.

In [ ]:
import re


def bump_patch(version: str) -> str:
    major, minor, patch = map(int, version.split("."))
    return f"{major}.{minor}.{patch + 1}"


current_text = (DEMO / "pyproject.toml").read_text(encoding="utf-8")
current_version = tomllib.loads(current_text)["project"]["version"]
next_version = bump_patch(current_version)
next_project = WORKDIR / "demo_pkg_project_next"
shutil.rmtree(next_project, ignore_errors=True)
shutil.copytree(DEMO, next_project)

updated_text = re.sub(
    rf'version = "{re.escape(current_version)}"',
    f'version = "{next_version}"',
    current_text,
    count=1,
)
(next_project / "pyproject.toml").write_text(updated_text, encoding="utf-8")

print("Current version:", current_version)
print("Next patch version:", next_version)
print("Updated file:", next_project / "pyproject.toml")
print("\nTypical release commands:")
print(f"- git tag v{next_version}")
print("- python -m build --sdist --wheel")
print("- python -m twine upload dist/*")

## 9. Private indexes and dependency sources

Not every package comes from the public PyPI index. Organizations often use:

- internal package indexes (Artifactory, Nexus, devpi, self-hosted simple indexes),
- wheelhouses served over HTTP or from a shared directory,
- constraints files for reproducible environments.

Important commands include:

- `--index-url` to set the primary index,
- `--extra-index-url` to add another index,
- `--find-links` to install from a directory or static page of wheels,
- `-c constraints.txt` to constrain transitive versions.

Be careful with `--extra-index-url`: it can create **dependency confusion** risks if package names overlap between public and private indexes.

In [ ]:
requirements_path = WORKDIR / "requirements.txt"
constraints_path = WORKDIR / "constraints.txt"

requirements_path.write_text(
    "demo-pkg-cursor==0.1.0\ninternal-lib>=2.0\n",
    encoding="utf-8",
)
constraints_path.write_text(
    "urllib3<3\nrequests!=2.32.0\n",
    encoding="utf-8",
)

print("requirements.txt:\n")
print(requirements_path.read_text(encoding="utf-8"))
print("constraints.txt:\n")
print(constraints_path.read_text(encoding="utf-8"))

print("Useful install commands:")
print("- python -m pip install -r requirements.txt -c constraints.txt")
print("- python -m pip install --index-url https://pypi.my-company.example/simple internal-lib")
print("- python -m pip install --find-links ./wheelhouse demo-pkg-cursor")
run([sys.executable, "-m", "pip", "config", "list"])

## 10. Managing releases: yank, replace, transfer, remove

Package maintenance does not stop after upload.

Important rules and operations:

- You generally **cannot replace** an existing uploaded version with different files.
- If a release is broken, prefer to **yank** it and publish a newer one.
- If a project changes hands, transfer or add **owners / maintainers** in the package index UI.
- If a project must disappear, removal rules depend on the index policy; full deletion is more limited than many newcomers expect.

Practical guidance:

- **Broken but still historically relevant** -> yank it.
- **Security fix** -> publish a new version quickly and document the issue.
- **Renaming or ownership changes** -> coordinate maintainers and update project metadata / README.
- **Accidental secret leak** -> rotate credentials first, then handle the package release itself.

In [ ]:
scenarios = {
    "broken wheel uploaded": "yank current release and upload a bumped version",
    "typo in README only": "publish a new patch release if the rendered metadata matters",
    "accidentally leaked token": "rotate the token immediately, then assess whether a new release or project cleanup is needed",
    "package moved to new team": "add / transfer maintainers and update documentation",
    "want to reuse the same version number": "do not do it; publish a new version instead",
}

for scenario, action in scenarios.items():
    print(f"- {scenario}: {action}")

print("\nPyPI operations such as yanking, ownership changes, and project deletion are usually performed in the web UI, not via pip itself.")